Exerciţiul 3. Detectarea unei schimbări graduale a ratei evenimentelor
Un sistem informatic înregistrează numărul zilnic de solicitări de asistenţă. Presupunem independenţa
condiţionată a observaţiilor şi modelul

$$ Yt | λt ∼ Poisson(λt), t = 1,...,30,     λt = λ1 + (λ2 − λ1) * \frac{1}{1 + e^{-k(t-r)}} $$

Parametrii λ1 şi λ2 reprezintă nivelurile iniţial şi final ale ratei, τ este momentul central al tranziţiei, iar
k > 0 controlează rapiditatea schimbării. Distribuţiile a priori sunt independente:
λ1, λ2 ∼ Exponential(0.2), τ ∼ Uniform(1, 30), k ∼ Exponential(1).

Datele observate sunt:

$$ [3, 4, 2, 5, 3, 4, 3, 2, 4, 3, 4, 5, 5, 6, 5, 7, 6, 8, 7, 9,
8, 10, 9, 8, 11, 9, 10, 8, 9, 10] $$

In [1]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm

# Date
y_obs = np.array([3, 4, 2, 5, 3, 4, 3, 2, 4, 3, 4, 5, 5, 6, 5, 7, 6, 8, 7, 9, 8, 10, 9, 8, 11, 9, 10, 8, 9, 10])
t = np.arange(1, len(y_obs) + 1)

a) Implementaţi modelul în PyMC şi generaţi eşantioane din distribuţiile a posteriori ale lui λ1, λ2, τ, k.

In [ ]:
# a) Modelul
with pm.Model() as model:
  lam1 = pm.Exponential("lam1", lam=0.2)
  lam2 = pm.Exponential("lam2", lam=0.2)
  tau = pm.Uniform("tau", lower=1.0, upper=30.0)
  k = pm.Exponential("k", lam=1.0)

  # Tranzitie graduală (sigmoida)
  lam_t = pm.Deterministic("lam_t", lam1 + (lam2 - lam1) / (1.0 + pm.math.exp(-k * (t - tau))))
  y = pm.Poisson("y", mu=lam_t, observed=y_obs)

  idata = pm.sample(draws=2000, tune=1000, chains=4, random_seed=42)

  # d) Distributia predictiva a posteriori
  pm.sample_posterior_predictive(idata, extend_inferencedata=True, random_seed=42)

summary = az.summary(idata, var_names=["lam1", "lam2", "tau", "k"], ci_kind="hdi", ci_prob=0.95)
print(summary)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [lam1, lam2, tau, k]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 5 seconds.
There were 8 divergences after tuning. Increase `target_accept` or reparameterize.
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [y]


Output()

      mean    sd hdi95_lb hdi95_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
lam1  2.88  1.06     0.46      4.6      763      389  1.01     0.045   0.037
lam2  10.1   2.5      6.7       16      331      188  1.01      0.22    0.36
tau   16.4     4      8.6       26      288      177  1.01      0.32    0.42
k     0.58  0.64    0.042      1.8      401      252  1.01     0.017   0.022


b) Reprezentaţi distribuţiile a posteriori ale lui τ şi k. Explicaţi semnificaţia parametrilor şi determinaţi
intervalele HDI de 95%.

In [8]:
az.plot_dist(idata, var_names=["tau", "k"], ci_kind="hdi", ci_prob=0.95, backend="matplotlib")
plt.suptitle("Distributiile a posteriori pentru τ și k (HDI 95%)", y=1.03)
plt.tight_layout()
plt.savefig("3b_tau_k.png", dpi=150)
plt.close()

c) Estimaţi P (λ2 > λ1 | Y1, . . . , Y30). Reprezentaţi evoluţia estimată a lui λt împreună cu datele observate.

In [13]:
lam1_s = idata.posterior["lam1"].values.flatten()
lam2_s = idata.posterior["lam2"].values.flatten()
prob_increase = np.mean(lam2_s > lam1_s)
print(f"\nP(lam2 > lam1 | Y_1:30) = {prob_increase:.4f}")

lam_t_s = idata.posterior["lam_t"].values.reshape(-1, len(t))
lam_mean = lam_t_s.mean(axis=0)

lam_hdi_da = az.hdi(idata.posterior, var_names=["lam_t"], prob=0.95)["lam_t"]
lam_hdi = lam_hdi_da.values

plt.figure(figsize=(9, 4.5))
plt.scatter(t, y_obs, color="black", label="Observatii $Y_t$", zorder=4)
plt.plot(t, lam_mean, color="tab:blue", lw=2, label="Media $\\lambda_t$")
plt.fill_between(
    t,
    lam_hdi[:, 0],
    lam_hdi[:, 1],
    color="tab:blue",
    alpha=0.25,
    label="HDI 95% pentru $\\lambda_t$",
)
plt.xlabel("Ziua ($t$)")
plt.ylabel("Solicitari / rata")
plt.title("Evoluaia ratei estimate $\\lambda_t$ și observatiile zilnice")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("3c_lambda_t.png", dpi=150)
plt.close()


P(lam2 > lam1 | Y_1:30) = 1.0000


d) Generaţi distribuţia predictivă a posteriori pentru cele 30 de zile şi comparaţi rezultatele cu datele
observate. Discutaţi în ce măsură modelul reproduce tranziţia graduală.

In [18]:
y_ppc_da = az.hdi(idata.posterior_predictive, var_names=["y"], prob=0.95)["y"]
ppc_hdi = y_ppc_da.values
ppc_mean = idata.posterior_predictive["y"].values.reshape(-1, len(t)).mean(axis=0)

plt.figure(figsize=(9, 4.5))
plt.scatter(t, y_obs, color="crimson", label="Observatii $Y_t$", zorder=4)
plt.plot(
    t, ppc_mean, color="tab:green", lw=2, label="Media predictiva a posteriori"
)
plt.fill_between(
    t,
    ppc_hdi[:, 0],
    ppc_hdi[:, 1],
    color="tab:green",
    alpha=0.25,
    label="HDI 95% predictiv",
)
plt.xlabel("Ziua ($t$)")
plt.ylabel("Număr de solicitări")
plt.title("Predictia a posteriori vs. datele observate")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("3d_ppc.png", dpi=150)
plt.close()